# Model: MLP (Neural Network)
Trains a Multi-Layer Perceptron that predicts **P(candidate road arc is on the true route)** for every (GPS point, candidate arc) row.

**Input:** `processed/{train,val,test}/model_data.npz` from `featureEngineering.ipynb`.
**Steps:** imports -> config -> load -> build model -> train with early stopping on val -> choose threshold -> evaluate (val and test) -> point-level accuracy -> save

## 1. Imports

In [1]:
import json
import os
import time

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import (accuracy_score, confusion_matrix, f1_score, log_loss,
                             precision_score, recall_score, roc_auc_score)
from sklearn.neural_network import MLPClassifier

## 2. Configuration
Hyperparameters of the network in one place (these are also what the Bayesian optimisation step can search over).
Early stopping is done on the **validation subjects**, not on a random slice of train rows, so the stopping signal is not leaked from the same subjects.

In [2]:
OUT_DIR = "processed"
SEED = 42

HIDDEN = (64, 32)        # hidden layer sizes
LR = 1e-3                # Adam learning rate
ALPHA = 1e-4             # L2 regularisation
BATCH_SIZE = 1024
MAX_EPOCHS = 40
PATIENCE = 5             # stop when val loss has not improved for this many epochs

os.makedirs(os.path.join(OUT_DIR, "models"), exist_ok=True)

## 3. Load data
`X` = scaled features, `y` = 1 if the candidate arc is on the route, `groups` = subject id.

In [3]:
def load(split):
    d = np.load(os.path.join(OUT_DIR, split, "model_data.npz"), allow_pickle=True)
    return {k: d[k] for k in d.files}

tr, va, te = load("train"), load("val"), load("test")
print("train", tr["X"].shape, "pos rate", tr["y"].mean().round(3))
print("val  ", va["X"].shape, "pos rate", va["y"].mean().round(3))
print("test ", te["X"].shape, "pos rate", te["y"].mean().round(3))
print("features:", [str(f) for f in tr["feature_names"]])

FileNotFoundError: [Errno 2] No such file or directory: 'processed\\train\\model_data.npz'

## 4. Build the model
`MLPClassifier`: ReLU hidden layers, Adam optimiser, binary cross-entropy output (sigmoid).
`partial_fit` runs exactly one shuffled pass over the data per call, which lets us monitor the validation set after every epoch.

In [ ]:
mlp = MLPClassifier(hidden_layer_sizes=HIDDEN, activation="relu", solver="adam",
                    learning_rate_init=LR, alpha=ALPHA, batch_size=BATCH_SIZE,
                    shuffle=True, random_state=SEED)

## 5. Train with early stopping
After each epoch we log train loss and val loss / F1, keep the best weights, and stop when val loss stops improving.

In [ ]:
import copy

history, best_loss, best_model, bad_epochs = [], np.inf, None, 0
t0 = time.time()
for epoch in range(1, MAX_EPOCHS + 1):
    mlp.partial_fit(tr["X"], tr["y"], classes=[0, 1])
    p_val = mlp.predict_proba(va["X"])[:, 1]
    v_loss = log_loss(va["y"], p_val)
    v_f1 = f1_score(va["y"], p_val >= 0.5)
    history.append(dict(epoch=epoch, train_loss=mlp.loss_, val_loss=v_loss, val_f1=v_f1))
    print(f"epoch {epoch:02d} | train loss {mlp.loss_:.4f} | val loss {v_loss:.4f} | val F1 {v_f1:.4f}")
    if v_loss < best_loss - 1e-4:
        best_loss, best_model, bad_epochs = v_loss, copy.deepcopy(mlp), 0
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("early stopping")
            break

mlp = best_model
history = pd.DataFrame(history)
print(f"trained in {time.time() - t0:.0f}s | best epoch {history.val_loss.idxmin() + 1} | best val loss {best_loss:.4f}")

epoch 01 | train loss 0.4034 | val loss 0.3963 | val F1 0.7348
epoch 02 | train loss 0.3658 | val loss 0.3944 | val F1 0.7332
epoch 03 | train loss 0.3539 | val loss 0.3951 | val F1 0.7340
epoch 04 | train loss 0.3464 | val loss 0.3959 | val F1 0.7333
epoch 05 | train loss 0.3411 | val loss 0.3977 | val F1 0.7320
epoch 06 | train loss 0.3372 | val loss 0.3991 | val F1 0.7316
epoch 07 | train loss 0.3342 | val loss 0.3998 | val F1 0.7307
early stopping
trained in 34s | best epoch 2 | best val loss 0.3944


## 6. Choose the decision threshold
0.5 is not always the best cut-off for an imbalanced target. We pick the threshold that maximises F1 on the **validation** set and reuse it on test.

In [ ]:
p_val = mlp.predict_proba(va["X"])[:, 1]
thresholds = np.arange(0.2, 0.81, 0.02)
f1s = [f1_score(va["y"], p_val >= t) for t in thresholds]
THRESHOLD = float(thresholds[int(np.argmax(f1s))])
print(f"best threshold {THRESHOLD:.2f} (val F1 {max(f1s):.4f}; at 0.50 it is {f1_score(va['y'], p_val >= 0.5):.4f})")

best threshold 0.40 (val F1 0.7409; at 0.50 it is 0.7332)


## 7. Evaluate on validation and test
Accuracy, precision, recall, F1, ROC-AUC and the confusion matrix (rows = actual, columns = predicted).

In [ ]:
def evaluate(d, name):
    p = mlp.predict_proba(d["X"])[:, 1]
    pred = (p >= THRESHOLD).astype(int)
    m = dict(split=name, accuracy=accuracy_score(d["y"], pred), precision=precision_score(d["y"], pred),
             recall=recall_score(d["y"], pred), f1=f1_score(d["y"], pred), roc_auc=roc_auc_score(d["y"], p),
             log_loss=log_loss(d["y"], p))
    cm = pd.DataFrame(confusion_matrix(d["y"], pred), index=["actual off-route", "actual on-route"],
                      columns=["pred off-route", "pred on-route"])
    return m, cm, p

m_val, cm_val, p_val = evaluate(va, "val")
m_test, cm_test, p_test = evaluate(te, "test")
metrics = pd.DataFrame([m_val, m_test]).round(4)
metrics

,split,accuracy,precision,recall,f1,roc_auc,log_loss
0,val,0.8226,0.7199,0.7631,0.7409,0.8827,0.3944
1,test,0.7905,0.7727,0.6766,0.7215,0.8230,0.5253


In [ ]:
print("Validation confusion matrix"); display(cm_val)
print("Test confusion matrix");       display(cm_test)

Validation confusion matrix


,pred off-route,pred on-route
actual off-route,178228,30922
actual on-route,24670,79487


Test confusion matrix


,pred off-route,pred on-route
actual off-route,89012,13685
actual on-route,22234,46516


## 8. Point-level accuracy (the metric that matters for map matching)
For each GPS point we take the candidate arc with the highest predicted probability and check whether it is on the true route
(`top-1 on-route accuracy`). This is what the HMM/Viterbi step later improves on by using the sequence.

In [ ]:
def point_accuracy(d, p):
    df = pd.DataFrame({"subject": d["groups"], "point": d["point_idx"], "p": p, "y": d["y"], "y_true": d["y_true_arc"]})
    top = df.loc[df.groupby(["subject", "point"])["p"].idxmax()]
    has_label = df.groupby(["subject", "point"])["y"].max()           # points where the true route is among the candidates
    covered = top.set_index(["subject", "point"]).index.map(has_label).to_numpy() == 1
    return dict(points=len(top), top1_on_route=float(top["y"].mean()),
                top1_on_route_covered_points=float(top.loc[covered, "y"].mean()),
                top1_exact_true_arc=float(top["y_true"].mean()))

point_metrics = pd.DataFrame([dict(split="val", **point_accuracy(va, p_val)),
                              dict(split="test", **point_accuracy(te, p_test))]).round(4)
point_metrics

,split,points,top1_on_route,top1_on_route_covered_points,top1_exact_true_arc
0,val,32259,0.8349,0.8383,0.6830
1,test,19158,0.8686,0.8693,0.7111


## 9. Save model, metrics and training history
The model file is what your API loads; the CSVs are evidence for the report.

In [ ]:
joblib.dump(dict(model=mlp, threshold=THRESHOLD, features=[str(f) for f in tr["feature_names"]]),
            os.path.join(OUT_DIR, "models", "mlp.joblib"))
metrics.to_csv(os.path.join(OUT_DIR, "models", "mlp_metrics.csv"), index=False)
point_metrics.to_csv(os.path.join(OUT_DIR, "models", "mlp_point_metrics.csv"), index=False)
history.to_csv(os.path.join(OUT_DIR, "models", "mlp_history.csv"), index=False)
with open(os.path.join(OUT_DIR, "models", "mlp_config.json"), "w") as f:
    json.dump(dict(hidden=HIDDEN, lr=LR, alpha=ALPHA, batch_size=BATCH_SIZE, threshold=THRESHOLD, seed=SEED), f, indent=1)
print("saved ->", os.listdir(os.path.join(OUT_DIR, "models")))

saved -> ['mlp.joblib', 'mlp_config.json', 'mlp_history.csv', 'mlp_metrics.csv', 'mlp_point_metrics.csv']
